# Arabic Classical ML Preprocessing — Fair Stemming Experiment

## Design Principle
This notebook implements a **single-split, dual-pipeline** preprocessing workflow to ensure
a fair comparison between stemmed and non-stemmed Arabic text classification experiments.

**The dataset is split exactly once.** Both pipelines reuse the same train/val/test indices.

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Install & import dependencies |
| 2 | Configuration block |
| 3 | Constants, regex patterns & stopwords |
| 4 | Helper preprocessing functions |
| 5 | Stemming function (ISRIStemmer) |
| 6 | Main preprocessing function (cleaning only, no stemming) |
| 7 | Dataset cleaning function |
| 8 | Load raw data |
| 9 | Run text cleaning → produce `clean_text` column |
| 10 | **Single dataset split** → save `train_raw`, `val_raw`, `test_raw` |
| 11 | Generate non-stemmed versions → save `train_clean`, `val_clean`, `test_clean` |
| 12 | Generate stemmed versions → save `train_stemmed`, `val_stemmed`, `test_stemmed` |
| 13 | Output summary |

---

## Output Files (`data/`)

```
data/
├── train_raw.csv       ← cleaned text, no stemming, train indices
├── val_raw.csv
├── test_raw.csv
├── train_clean.csv     ← non-stemmed pipeline (same indices as raw)
├── val_clean.csv
├── test_clean.csv
├── train_stemmed.csv   ← stemmed pipeline (same indices as raw)
├── val_stemmed.csv
└── test_stemmed.csv
```

---
## 1 · Install & Import Dependencies

In [1]:
!pip install emoji deep-translator nltk scikit-learn pandas numpy joblib openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.8 MB/s eta 0:00:00


In [2]:
import os
import re
import warnings
import unicodedata
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import nltk
nltk.download("stopwords", quiet=True)
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords

# Optional: emoji library
try:
    import emoji as emoji_lib
    EMOJI_OK = True
except ImportError:
    EMOJI_OK = False
    print("[WARN] `emoji` not installed. Emoji handling will use regex fallback.")

# Optional: deep_translator for English→Arabic translation
try:
    from deep_translator import GoogleTranslator
    TRANSLATOR_OK = True
except ImportError:
    TRANSLATOR_OK = False
    print("[WARN] `deep_translator` not installed. English→Arabic translation disabled.")

print("[OK] All imports complete.")

[OK] All imports complete.


---
## 2 · Configuration Block

Edit the values in this cell before running the notebook.

In [3]:
# ============================================================
# CONFIGURATION — edit these values before running
# ============================================================

# --- Paths ---
DATASET_PATH   = "/content/Multi-dialect Arabic dataset.xlsx"  # path to raw data
SHEET_INDEX    = 17          # Excel sheet index (0-based)
OUTPUT_DIR     = "data"      # all output CSVs will be written here

# --- Column names in the raw dataset ---
TEXT_COL       = "comment"   # column containing raw text
LABEL_COL      = "label"     # column containing class labels

# --- Preprocessing toggles ---
NORMALIZE_TA_MARBUTA = True  # ة → ه
NORMALIZE_HAMZA      = True  # ؤ / ئ → ء
NORMALIZE_PERSIAN    = True  # چ→ج  گ→ك  پ→ب  ڤ→ف  ې→ي
REMOVE_EMOJIS        = True  # remove emoji characters (count is still saved)
REMOVE_NON_ARABIC    = True  # strip non-Arabic / non-digit characters
TRANSLATE_ENGLISH    = False # translate Latin tokens to Arabic (slow — Google API)
DROP_NEUTRAL_LABEL   = True  # drop rows whose label is 'neutral'

# --- Split ratios ---
TEST_SIZE      = 0.15        # fraction of data held out as test set
VAL_SIZE       = 0.15        # fraction of data held out as validation set
RANDOM_STATE   = 42          # fixed seed — guarantees reproducibility

# --- Output column names ---
CLEAN_COL      = "clean_text"         # column name for cleaned (unstemmed) text
STEMMED_COL    = "clean_text_stemmed" # column name for stemmed text

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[OK] Configuration loaded.")
print(f"     OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"     TEST_SIZE    = {TEST_SIZE}")
print(f"     VAL_SIZE     = {VAL_SIZE}")
print(f"     RANDOM_STATE = {RANDOM_STATE}")

[OK] Configuration loaded.
     OUTPUT_DIR   = data
     TEST_SIZE    = 0.15
     VAL_SIZE     = 0.15
     RANDOM_STATE = 42


---
## 3 · Constants, Regex Patterns & Stopwords

In [4]:
# ── Numeric digit normalization ───────────────────────────────────────────────
# Arabic-Indic (٠١٢…) and Extended Arabic-Indic (۰۱۲…) → Western digits (012…)
_AR_INDIC     = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
_EXT_AR_INDIC = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")

# Arabizi digit→letter substitutions common in social media
ARABIZI_DIGIT_MAP = {
    "2": "ء", "3": "ع", "5": "خ", "6": "ط",
    "7": "ح", "8": "ق", "9": "ص",
}

# ── Persian letter normalization ──────────────────────────────────────────────
PERSIAN_NORM_MAP = str.maketrans({
    "چ": "ج",   # Persian che → Arabic jeem
    "گ": "ك",   # Persian gaf → Arabic kaf
    "پ": "ب",   # Persian pe  → Arabic ba
    "ڤ": "ف",   # Persian va  → Arabic fa
    "ې": "ي",   # Pashto ye   → Arabic ya
})

# ── Ligature normalization map ────────────────────────────────────────────────
LIGATURE_MAP = str.maketrans({
    "ﷲ": "الله",
    "ﻷ": "لا",
    "ﻹ": "لا",
    "ﻵ": "لا",
    "ﻻ": "لا",
})

# ── Compiled regex patterns ───────────────────────────────────────────────────
RE_URL       = re.compile(r"https?://\S+|www\.\S+")
RE_MENTION   = re.compile(r"@\w+")
RE_HASHTAG   = re.compile(r"#(\w+)")           # keep the hashtag word, drop '#'
RE_FLAG      = re.compile(r"[\U0001F1E6-\U0001F1FF]{2}")
RE_EMOJI_FB  = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)
RE_TASHKEEL  = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
RE_TATWEEL   = re.compile(r"\u0640")
RE_OBFUSC_AR = re.compile(r"(?<=[\u0600-\u06FF])([^\u0600-\u06FF\w\s]+)(?=[\u0600-\u06FF])")
RE_ELONGATE  = re.compile(r"(.)\1{2,}")
RE_LATIN     = re.compile(r"^[A-Za-z]{2,}$")
RE_AR_TOKEN  = re.compile(r"[\u0600-\u06FF]+|\d+")
RE_REPUNCT   = re.compile(r"([!?.,،؛؟]){2,}")

# ── Arabic stopwords (NLTK base + dialectal additions) ───────────────────────
try:
    AR_STOPWORDS = set(stopwords.words("arabic"))
except Exception:
    AR_STOPWORDS = set()

# Common dialectal stopwords not in the NLTK list
_DIALECT_STOPS = {
    "يعني", "هيك", "هكذا", "هكي", "كذا", "زي", "زيي", "ذي",
    "هذا", "هاذا", "هاد", "هاي", "واحد", "شي", "حاجة",
    "لما", "لمن", "لحد", "ومن", "وفي", "وعلى",
    "يا", "يالي", "اللي", "إللي", "لي", "الي",
    "فيه", "فيها", "فيهم", "منه", "منها", "منهم", "عليه", "عليها",
}
AR_STOPWORDS.update(_DIALECT_STOPS)

# ── ISRIStemmer instance ──────────────────────────────────────────────────────
stemmer = ISRIStemmer()

print(f"[OK] Constants loaded. Arabic stopwords count: {len(AR_STOPWORDS)}")

[OK] Constants loaded. Arabic stopwords count: 722


---
## 4 · Helper Preprocessing Functions

Each function handles one specific cleaning operation.
They are composable and called in a fixed order by the main pipeline.

In [5]:
# ── Unicode & digit normalization ─────────────────────────────────────────────

def unicode_normalize(text: str) -> str:
    """Apply NFC unicode normalization and convert Arabic-Indic digits to Western."""
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(_AR_INDIC)
    text = text.translate(_EXT_AR_INDIC)
    return text


def convert_arabic_indic(text: str) -> str:
    """Convert Arabic-Indic and Extended Arabic-Indic numerals to Western digits."""
    return text.translate(_AR_INDIC).translate(_EXT_AR_INDIC)


def normalize_ligatures(text: str) -> str:
    """Expand common Arabic ligatures (ﷲ → الله, ﻻ → لا, etc.)."""
    return text.translate(LIGATURE_MAP)


# ── Emoji & flag handling ─────────────────────────────────────────────────────

def extract_flag_codes(text: str):
    """Extract regional indicator flag sequences and return (cleaned_text, flags_list)."""
    flags = RE_FLAG.findall(text)
    text  = RE_FLAG.sub(" ", text)
    return text, flags


def extract_emojis(text: str):
    """Remove emojis and return (cleaned_text, emoji_count).

    Uses the `emoji` library when available; falls back to a broad Unicode regex.
    """
    if EMOJI_OK:
        emoji_chars = [c for c in text if c in emoji_lib.EMOJI_DATA]
        count = len(emoji_chars)
        text  = emoji_lib.replace_emoji(text, replace=" ")
    else:
        matches = RE_EMOJI_FB.findall(text)
        count   = len(matches)
        text    = RE_EMOJI_FB.sub(" ", text)
    return text, count


# ── URL, mention & hashtag handling ──────────────────────────────────────────

def remove_urls_mentions(text: str) -> str:
    """Remove HTTP/HTTPS URLs and @mention tokens."""
    return RE_MENTION.sub(" ", RE_URL.sub(" ", text))


def clean_hashtags(text: str) -> str:
    """Strip the '#' symbol but keep the hashtag word as a regular token."""
    return RE_HASHTAG.sub(r" \1 ", text)


# ── Diacritics & elongation ───────────────────────────────────────────────────

def remove_tashkeel(text: str) -> str:
    """Remove Arabic diacritics (tashkeel) and the tatweel elongation character."""
    return RE_TATWEEL.sub("", RE_TASHKEEL.sub("", text))


def collapse_elongation(text: str) -> str:
    """Reduce any character repeated 3+ times to at most 2 repetitions."""
    return RE_ELONGATE.sub(r"\1\1", text)


# ── Arabic letter normalization ───────────────────────────────────────────────

def normalize_arabic_letters(text: str,
                              ta_marbuta: bool = True,
                              hamza: bool = True,
                              persian: bool = True) -> str:
    """Normalize Arabic letter variants.

    - Alef variants (إأٱآ) → ا
    - Alef Maqsura (ى) → ي
    - Ta Marbuta (ة) → ه   [configurable via ta_marbuta]
    - Hamza variants (ؤئ) → ء [configurable via hamza]
    - Persian letters (چگپڤې) → Arabic equivalents [configurable via persian]
    """
    text = re.sub(r"[إأٱآ]", "ا", text)       # Alef variants
    text = re.sub(r"ى",      "ي", text)        # Alef Maqsura
    if ta_marbuta:
        text = re.sub(r"ة", "ه", text)         # Ta Marbuta
    if hamza:
        text = re.sub(r"[ؤئ]", "ء", text)     # Hamza forms
    if persian:
        text = text.translate(PERSIAN_NORM_MAP) # Persian → Arabic equivalents
    return text


# ── Obfuscation & fragmentation repair ───────────────────────────────────────

def deobfuscate(text: str) -> str:
    """Remove punctuation/separators injected between Arabic letters (obfuscation)."""
    prev = None
    while prev != text:
        prev = text
        text = RE_OBFUSC_AR.sub("", text)
    return text


def fix_fragmented_arabic(text: str) -> str:
    """Merge runs of isolated single Arabic characters (e.g. 'ت ا غ ي ر' → 'تاغير')."""
    tokens = text.split(" ")
    result, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        if len(tok) == 1 and re.match(r"[\u0600-\u06FF]", tok):
            run = [tok]
            j = i + 1
            while j < len(tokens) and len(tokens[j]) == 1 \
                    and re.match(r"[\u0600-\u06FF]", tokens[j]):
                run.append(tokens[j])
                j += 1
            result.append("".join(run) if len(run) >= 2 else tok)
            i = j if len(run) >= 2 else i + 1
        else:
            result.append(tok)
            i += 1
    return " ".join(result)


# ── Arabizi handling ──────────────────────────────────────────────────────────

def handle_arabizi_tokens(text: str) -> str:
    """Convert Arabizi digit-letter tokens using a common social-media substitution map."""
    tokens, result = text.split(), []
    for tok in tokens:
        if tok.isdigit():
            result.append(tok)
        elif re.search(r"\d", tok) and re.search(r"[A-Za-z\u0600-\u06FF]", tok):
            result.append("".join(ARABIZI_DIGIT_MAP.get(ch, ch) for ch in tok))
        else:
            result.append(tok)
    return " ".join(result)


def remove_non_arabic_digits(text: str) -> str:
    """Keep only Arabic script characters and digits; strip everything else."""
    text = re.sub(r"[^\u0600-\u06FF0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


# ── English→Arabic batch translation helper ───────────────────────────────────

def batch_translate_english(text_series: pd.Series, translate: bool = True) -> dict:
    """Build a lookup dict {english_word: arabic_translation} from the corpus.

    Only active when translate=True AND deep_translator is installed.
    Returns an empty dict otherwise.
    """
    if not translate or not TRANSLATOR_OK:
        return {}
    print("[INFO] Extracting unique English words for batch translation …")
    all_english = {
        w.lower() for text in text_series
        for w in str(text).split() if RE_LATIN.match(w)
    }
    lookup, batch_size = {}, 50
    word_list = list(all_english)
    for i in range(0, len(word_list), batch_size):
        batch = word_list[i: i + batch_size]
        try:
            joined     = " | ".join(batch)
            translated = GoogleTranslator(source="auto", target="ar").translate(joined)
            for w, t in zip(batch, translated.split("|")):
                lookup[w] = t.strip()
        except Exception:
            pass
    print(f"[INFO] Translated {len(lookup)} unique English terms.")
    return lookup


def apply_translation_lookup(text: str, lookup: dict) -> str:
    """Replace recognised Latin tokens with their Arabic translations."""
    tokens = text.split()
    return " ".join(
        lookup.get(tok.lower(), tok) if RE_LATIN.match(tok) else tok
        for tok in tokens
    )


print("[OK] Helper preprocessing functions defined.")

[OK] Helper preprocessing functions defined.


---
## 5 · Stemming Function (ISRIStemmer)

This function is applied **only** to the stemmed pipeline in Step 12.  
It is **not** called during the base cleaning step, ensuring the non-stemmed
dataset is a true control condition.

In [6]:
def stem_and_remove_stopwords(text: str) -> str:
    """Tokenize Arabic text, remove stopwords, and apply ISRIStemmer.

    Rules:
    - Pure digit tokens → kept unchanged.
    - Stopword tokens   → dropped.
    - All other Arabic tokens → stemmed via ISRIStemmer.

    Note: stopword removal is applied here (and only here) so that the
    non-stemmed pipeline is NOT filtered by stopwords, making the
    stemming step the only variable between the two datasets.
    If you want stopword removal in both pipelines, move this logic
    into `clean_text_row` and call it before the split.
    """
    tokens    = RE_AR_TOKEN.findall(text)
    processed = []
    for token in tokens:
        if token.isdigit():
            processed.append(token)           # keep numbers as-is
        elif token in AR_STOPWORDS:
            continue                           # drop stopwords
        else:
            try:
                processed.append(stemmer.stem(token))
            except Exception:
                processed.append(token)        # fallback: keep original
    return " ".join(processed)


# ── Quick smoke-test ──────────────────────────────────────────────────────────
print("[OK] Stemming function defined (ISRIStemmer).")
_test = "المتعلمون يكتبون الدروس في الفصل"
_stemmed = " ".join(stemmer.stem(w) for w in _test.split())
print(f"  Input   : {_test}")
print(f"  Stemmed : {_stemmed}")

[OK] Stemming function defined (ISRIStemmer).
  Input   : المتعلمون يكتبون الدروس في الفصل
  Stemmed : تعلم كتب درس في فصل


---
## 6 · Main Text Cleaning Function

Applies all normalization and cleaning steps **without stemming**.
Stemming is a separate, downstream operation (Step 12).

In [7]:
def clean_text_row(text: str, translation_lookup: dict) -> tuple:
    """Full per-row cleaning pipeline — NO stemming.

    Returns:
        clean_text  (str)  : cleaned text, ready for feature extraction or stemming.
        emoji_count (int)  : number of emoji characters found in the original text.

    Pipeline order:
        unicode normalize → Arabic-Indic digits → ligatures → flags →
        emojis → URLs/mentions → hashtags → translation lookup →
        deobfuscate → fix fragments → tashkeel removal →
        letter normalization (incl. Persian) → elongation collapse →
        Arabizi handling → non-Arabic strip → punctuation collapse →
        final whitespace normalization
    """
    # 1. Unicode & numeral normalization
    text = unicode_normalize(text)
    text = convert_arabic_indic(text)

    # 2. Ligature expansion
    text = normalize_ligatures(text)

    # 3. Flag removal
    text, _flags = extract_flag_codes(text)

    # 4. Emoji handling (count + optional removal)
    text, emoji_count = extract_emojis(text)
    if not REMOVE_EMOJIS:
        # If we are NOT removing emojis, restore them as a placeholder token
        # (the count is already captured; this branch keeps the flow clean)
        pass

    # 5. URL and @mention removal
    text = remove_urls_mentions(text)

    # 6. Hashtag processing — keep the word, drop the '#'
    text = clean_hashtags(text)

    # 7. English→Arabic translation (no-op when disabled)
    text = apply_translation_lookup(text, translation_lookup)

    # 8. Deobfuscation
    text = deobfuscate(text)

    # 9. Fragment repair
    text = fix_fragmented_arabic(text)

    # 10. Diacritics & tatweel removal
    text = remove_tashkeel(text)

    # 11. Arabic letter normalization (Alef, Ya, Ta Marbuta, Hamza, Persian)
    text = normalize_arabic_letters(
        text,
        ta_marbuta=NORMALIZE_TA_MARBUTA,
        hamza=NORMALIZE_HAMZA,
        persian=NORMALIZE_PERSIAN,
    )

    # 12. Repeated letter reduction (e.g. وااااو → واو)
    text = collapse_elongation(text)

    # 13. Arabizi digit-letter substitution
    text = handle_arabizi_tokens(text)

    # 14. Strip non-Arabic / non-digit characters
    if REMOVE_NON_ARABIC:
        text = remove_non_arabic_digits(text)

    # 15. Collapse repeated punctuation and normalize whitespace
    text = RE_REPUNCT.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text, emoji_count


print("[OK] clean_text_row defined (cleaning only, no stemming).")

[OK] clean_text_row defined (cleaning only, no stemming).


---
## 7 · Dataset Cleaning Function

Loads and cleans the entire dataset, producing a `clean_text` column.
**No stemming is applied here.**  Stemming is handled later in Step 12.

In [8]:
def clean_dataset(df: pd.DataFrame,
                  text_col:     str  = TEXT_COL,
                  label_col:    str  = LABEL_COL,
                  translate:    bool = TRANSLATE_ENGLISH,
                  drop_neutral: bool = DROP_NEUTRAL_LABEL) -> pd.DataFrame:
    """Clean the full dataset and produce the base `clean_text` column.

    Steps:
        0. Log initial shape and label distribution.
        1. Drop duplicate rows (by text column).
        2. Optionally drop rows with label == 'neutral'.
        3. Drop rows with empty text.
        4. Build English→Arabic translation lookup (once, for efficiency).
        5. Apply per-row text cleaning.
        6. Add metadata columns: emoji_count, has_emoji, has_hashtag, text_length.
        7. Remove rows where cleaning produced empty text.
        8. Encode labels to integers.
    """
    df = df.copy()
    df[text_col] = df[text_col].fillna("").astype(str)

    # ── Step 0: Initial statistics ────────────────────────────────────────────
    print(f"[STEP 0] Dataset shape before cleaning : {df.shape}")
    print(f"         Label distribution:\n{df[label_col].value_counts()}\n")

    # ── Step 1: Drop duplicates ───────────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=[text_col])
    print(f"[STEP 1] Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

    # ── Step 2: Drop neutral label ────────────────────────────────────────────
    if drop_neutral and "neutral" in df[label_col].str.lower().unique():
        df = df[df[label_col].str.lower() != "neutral"].reset_index(drop=True)
        print(f"[STEP 2] Dropped neutral labels. Remaining: {len(df)}")

    # ── Step 3: Drop rows with empty text ─────────────────────────────────────
    df = df[df[text_col].str.strip() != ""].reset_index(drop=True)
    print(f"[STEP 3] After dropping empty-text rows. Remaining: {len(df)}")

    # ── Step 4: Build translation lookup once ────────────────────────────────
    translation_lookup = batch_translate_english(df[text_col], translate=translate)

    # ── Step 5: Per-row text cleaning ─────────────────────────────────────────
    print("[STEP 5] Cleaning text (no stemming) — this may take a moment …")
    clean_texts, emoji_counts, hashtag_flags = [], [], []

    for raw_text in df[text_col]:
        # Capture hashtag presence BEFORE cleaning strips the '#'
        hashtag_flags.append(int("#" in unicodedata.normalize("NFC", str(raw_text))))
        cleaned, em_cnt = clean_text_row(raw_text, translation_lookup)
        clean_texts.append(cleaned)
        emoji_counts.append(em_cnt)

    # ── Step 6: Attach metadata columns ──────────────────────────────────────
    df[CLEAN_COL]        = clean_texts
    df["emoji_count"]    = emoji_counts
    df["has_emoji"]      = (df["emoji_count"] > 0).astype(int)
    df["has_hashtag"]    = hashtag_flags
    df["text_length"]    = df[CLEAN_COL].apply(lambda x: len(x.split()))

    # ── Step 7: Remove rows where cleaning produced empty text ────────────────
    empty_mask = df[CLEAN_COL].str.strip() == ""
    print(f"[STEP 7] Removed {empty_mask.sum()} rows with empty text after cleaning.")
    df = df[~empty_mask].reset_index(drop=True)

    # ── Step 8: Label encoding ────────────────────────────────────────────────
    le = LabelEncoder()
    df["labels"] = le.fit_transform(df[label_col])
    id2label = dict(enumerate(le.classes_))
    print(f"[STEP 8] Label encoding: {id2label}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n[INFO] Final cleaned dataset shape : {df.shape}")
    print(f"[INFO] Label distribution after preprocessing:\n{df[label_col].value_counts()}\n")

    # Log optional meta columns if present
    for meta_col in ["dialect", "platform"]:
        if meta_col in df.columns:
            print(f"[INFO] {meta_col.capitalize()} distribution:\n{df[meta_col].value_counts()}\n")

    # Preview 3 examples
    print("[INFO] Sample cleaning results (first 3 rows):")
    for i, (orig, clean) in enumerate(
            zip(df[text_col].head(3), df[CLEAN_COL].head(3)), 1):
        print(f"\n  [{i}] Original : {orig[:80]}")
        print(f"       Cleaned  : {clean[:80]}")

    return df, id2label


print("[OK] clean_dataset function defined.")

[OK] clean_dataset function defined.


---
## 8 · Load Raw Data

In [9]:
# Load dataset from Excel (or CSV — swap read_excel for read_csv if needed)
df_raw = pd.read_excel(DATASET_PATH, sheet_name=SHEET_INDEX)

print(f"[OK] Loaded dataset: {df_raw.shape}")
print(f"     Columns : {list(df_raw.columns)}")
display(df_raw.head(3))

[OK] Loaded dataset: (7627, 5)
     Columns : ['comment', 'dialect', 'platform', 'category', 'label']


,comment,dialect,platform,category,label
0,ماشاء الله يا حظج ويا هناج والله سلمى عليهم وق...,Kuwait,X,Feminist,positive
1,بناتنا الجميلات 😍😍 حبيتهم وايد والشغل معاهم مت...,Kuwait,X,Feminist,positive
2,حبيبتى الله يونسج بالعافيه وحمدالله على سلامتج,Kuwait,X,Feminist,positive


---
## 9 · Run Text Cleaning → `clean_text` Column

This step produces a single cleaned dataset that serves as the **shared base**
for both the non-stemmed and the stemmed pipelines.

In [10]:
# Run cleaning — no stemming is applied at this stage
df_cleaned, id2label = clean_dataset(df_raw)

# Keep only the columns needed for downstream experiments
BASE_COLS = ["labels", CLEAN_COL, "emoji_count", "has_emoji", "has_hashtag", "text_length"]
df_cleaned = df_cleaned[BASE_COLS].copy()

print(f"\n[OK] Cleaned dataset shape : {df_cleaned.shape}")
display(df_cleaned.head(3))

[STEP 0] Dataset shape before cleaning : (7627, 5)
         Label distribution:
label
positive    3627
negative    3625
Neutral      371
Name: count, dtype: int64

[STEP 1] Dropped 19 duplicate rows. Remaining: 7608
[STEP 2] Dropped neutral labels. Remaining: 7239
[STEP 3] After dropping empty-text rows. Remaining: 7238
[STEP 5] Cleaning text (no stemming) — this may take a moment …
[STEP 7] Removed 1 rows with empty text after cleaning.
[STEP 8] Label encoding: {0: 'negative', 1: 'positive'}

[INFO] Final cleaned dataset shape : (7237, 11)
[INFO] Label distribution after preprocessing:
label
positive    3621
negative    3616
Name: count, dtype: int64

[INFO] Dialect distribution:
dialect
Algeria      528
Yemen        528
Jordan       528
Sudanese     528
Libya        528
Saudi        528
Iraq         527
Egypt        527
Palestine    526
Moroccan     526
Lebanon      525
Syria        525
Tunisia      524
Kuwait       132
Emirates     132
Bahrain       50
Qaṭar         40
Oman         

,labels,clean_text,emoji_count,has_emoji,has_hashtag,text_length
0,1,ماشاء الله يا حظج ويا هناج والله سلمي عليهم وق...,1,1,0,14
1,1,بناتنا الجميلات حبيتهم وايد والشغل معاهم متعه,3,1,0,7
2,1,حبيبتي الله يونسج بالعافيه وحمدالله علي سلامتج,0,0,0,7


---
## 10 · Single Dataset Split

> **This is the only place in the notebook where the data is split.**
> Both the non-stemmed and the stemmed pipelines reuse exactly these indices.

Strategy: stratified two-phase split
1. Carve out the **test** set (15 %).
2. Carve out the **validation** set from the remainder (15 % of total ≈ 17.6 % of remainder).
3. Everything left becomes the **training** set.

In [11]:
y = df_cleaned["labels"].values

# ── Phase 1: carve out the test set ──────────────────────────────────────────
df_temp, df_test, y_temp, y_test = train_test_split(
    df_cleaned, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# ── Phase 2: carve out the validation set from the remaining data ─────────────
relative_val = VAL_SIZE / (1.0 - TEST_SIZE)
df_train, df_val, _, _ = train_test_split(
    df_temp, y_temp,
    test_size=relative_val,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

# Reset indices so each CSV has clean 0-based row numbers
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# ── Print split statistics ────────────────────────────────────────────────────
total = len(df_train) + len(df_val) + len(df_test)
print(f"[SPLIT] Train      : {len(df_train):>5} rows  ({len(df_train)/total*100:.1f}%)")
print(f"[SPLIT] Validation : {len(df_val):>5} rows  ({len(df_val)/total*100:.1f}%)")
print(f"[SPLIT] Test       : {len(df_test):>5} rows  ({len(df_test)/total*100:.1f}%)")
print(f"[SPLIT] Total      : {total:>5} rows")

print("\n[INFO] Label distribution per split:")
for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    counts = split_df["labels"].value_counts().sort_index()
    dist   = {id2label[k]: v for k, v in counts.items()}
    print(f"  {name:5s}: {dist}")

# ── Save raw splits (cleaned text, no stemming) ───────────────────────────────
raw_train_path = os.path.join(OUTPUT_DIR, "train_raw.csv")
raw_val_path   = os.path.join(OUTPUT_DIR, "val_raw.csv")
raw_test_path  = os.path.join(OUTPUT_DIR, "test_raw.csv")

df_train.to_csv(raw_train_path, index=False, encoding="utf-8-sig")
df_val.to_csv(  raw_val_path,   index=False, encoding="utf-8-sig")
df_test.to_csv( raw_test_path,  index=False, encoding="utf-8-sig")

print(f"\n[SAVED] train_raw.csv ({len(df_train)} rows)  → {raw_train_path}")
print(f"[SAVED] val_raw.csv   ({len(df_val)} rows)  → {raw_val_path}")
print(f"[SAVED] test_raw.csv  ({len(df_test)} rows)  → {raw_test_path}")

[SPLIT] Train      :  5065 rows  (70.0%)
[SPLIT] Validation :  1086 rows  (15.0%)
[SPLIT] Test       :  1086 rows  (15.0%)
[SPLIT] Total      :  7237 rows

[INFO] Label distribution per split:
  Train: {'negative': 2530, 'positive': 2535}
  Val  : {'negative': 543, 'positive': 543}
  Test : {'negative': 543, 'positive': 543}

[SAVED] train_raw.csv (5065 rows)  → data/train_raw.csv
[SAVED] val_raw.csv   (1086 rows)  → data/val_raw.csv
[SAVED] test_raw.csv  (1086 rows)  → data/test_raw.csv


---
## 11 · Generate Non-Stemmed Dataset

The `clean_text` column is renamed to `clean_text` in the output files.
No additional transformation is applied — this is the control condition.

In [12]:
# Re-use the already-split DataFrames; simply rename the text column for clarity
def make_clean_split(df_split: pd.DataFrame) -> pd.DataFrame:
    """Select and rename columns for the non-stemmed output file."""
    out = df_split[["labels", CLEAN_COL, "emoji_count", "has_emoji"]].copy()
    return out


train_clean = make_clean_split(df_train)
val_clean   = make_clean_split(df_val)
test_clean  = make_clean_split(df_test)

clean_train_path = os.path.join(OUTPUT_DIR, "train_clean.csv")
clean_val_path   = os.path.join(OUTPUT_DIR, "val_clean.csv")
clean_test_path  = os.path.join(OUTPUT_DIR, "test_clean.csv")

train_clean.to_csv(clean_train_path, index=False, encoding="utf-8-sig")
val_clean.to_csv(  clean_val_path,   index=False, encoding="utf-8-sig")
test_clean.to_csv( clean_test_path,  index=False, encoding="utf-8-sig")

print("[SAVED] Non-stemmed splits:")
print(f"  train_clean.csv ({len(train_clean)} rows) → {clean_train_path}")
print(f"  val_clean.csv   ({len(val_clean)} rows) → {clean_val_path}")
print(f"  test_clean.csv  ({len(test_clean)} rows) → {clean_test_path}")

print("\n[PREVIEW] First 3 rows of train_clean:")
display(train_clean.head(3))

[SAVED] Non-stemmed splits:
  train_clean.csv (5065 rows) → data/train_clean.csv
  val_clean.csv   (1086 rows) → data/val_clean.csv
  test_clean.csv  (1086 rows) → data/test_clean.csv

[PREVIEW] First 3 rows of train_clean:


,labels,clean_text,emoji_count,has_emoji
0,1,عفيه عليكم النشاما الحدود,0,0
1,1,اشتركت في قناتك لاني عمانيه وعجبني كلامك,0,0
2,1,الحب العاشق لي المصررين,2,1


---
## 12 · Generate Stemmed Dataset

Apply `stem_and_remove_stopwords` to the **already-split** DataFrames.
The row indices are identical to those in the non-stemmed files — the only
difference is the text representation.

In [13]:
def apply_stemming_to_split(df_split: pd.DataFrame) -> pd.DataFrame:
    """Apply stemming + stopword removal to the clean_text column.

    Returns a new DataFrame with a `clean_text_stemmed` column.
    The input DataFrame is NOT modified.
    """
    out = df_split.copy()
    print(f"  Stemming {len(out)} rows …", end="", flush=True)
    out[STEMMED_COL] = out[CLEAN_COL].apply(stem_and_remove_stopwords)
    print(" done.")
    # Keep only the stemmed text column (drop the unstemmed one)
    return out[["labels", STEMMED_COL, "emoji_count", "has_emoji"]]


print("[INFO] Applying stemming to train/val/test splits …")
train_stemmed = apply_stemming_to_split(df_train)
val_stemmed   = apply_stemming_to_split(df_val)
test_stemmed  = apply_stemming_to_split(df_test)

stemmed_train_path = os.path.join(OUTPUT_DIR, "train_stemmed.csv")
stemmed_val_path   = os.path.join(OUTPUT_DIR, "val_stemmed.csv")
stemmed_test_path  = os.path.join(OUTPUT_DIR, "test_stemmed.csv")

train_stemmed.to_csv(stemmed_train_path, index=False, encoding="utf-8-sig")
val_stemmed.to_csv(  stemmed_val_path,   index=False, encoding="utf-8-sig")
test_stemmed.to_csv( stemmed_test_path,  index=False, encoding="utf-8-sig")

print("\n[SAVED] Stemmed splits:")
print(f"  train_stemmed.csv ({len(train_stemmed)} rows) → {stemmed_train_path}")
print(f"  val_stemmed.csv   ({len(val_stemmed)} rows) → {stemmed_val_path}")
print(f"  test_stemmed.csv  ({len(test_stemmed)} rows) → {stemmed_test_path}")

print("\n[PREVIEW] First 3 rows of train_stemmed:")
display(train_stemmed.head(3))

[INFO] Applying stemming to train/val/test splits …
  Stemming 5065 rows … done.
  Stemming 1086 rows … done.
  Stemming 1086 rows … done.

[SAVED] Stemmed splits:
  train_stemmed.csv (5065 rows) → data/train_stemmed.csv
  val_stemmed.csv   (1086 rows) → data/val_stemmed.csv
  test_stemmed.csv  (1086 rows) → data/test_stemmed.csv

[PREVIEW] First 3 rows of train_stemmed:


,labels,clean_text_stemmed,emoji_count,has_emoji
0,1,عفه علي نشا حدد,0,0
1,1,شرك قنت لني عمن عجب كلم,0,0
2,1,لحب عشق صرر,2,1


---
## 13 · Output Summary

Final verification: confirm all 9 files exist and have matching row counts.

In [14]:
# ── Row-count verification ────────────────────────────────────────────────────
# The raw, clean, and stemmed versions of each split must have identical row counts.

file_groups = [
    ("train_raw",     raw_train_path,     df_train),
    ("val_raw",       raw_val_path,       df_val),
    ("test_raw",      raw_test_path,      df_test),
    ("train_clean",   clean_train_path,   train_clean),
    ("val_clean",     clean_val_path,     val_clean),
    ("test_clean",    clean_test_path,    test_clean),
    ("train_stemmed", stemmed_train_path, train_stemmed),
    ("val_stemmed",   stemmed_val_path,   val_stemmed),
    ("test_stemmed",  stemmed_test_path,  test_stemmed),
]

print("=" * 60)
print("  OUTPUT FILE SUMMARY")
print("=" * 60)
print(f"{'File':<22}  {'Rows':>6}  {'Size (KB)':>10}  {'Status'}")
print("-" * 60)

all_ok = True
for name, path, df_ref in file_groups:
    exists = os.path.isfile(path)
    if exists:
        size_kb = os.path.getsize(path) / 1024
        on_disk = pd.read_csv(path, encoding="utf-8-sig")
        row_match = len(on_disk) == len(df_ref)
        status = "✓ OK" if row_match else "✗ ROW MISMATCH"
        if not row_match:
            all_ok = False
        print(f"  {name:<20}  {len(on_disk):>6}  {size_kb:>10.1f}  {status}")
    else:
        all_ok = False
        print(f"  {name:<20}  {'—':>6}  {'—':>10}  ✗ FILE NOT FOUND")

print("-" * 60)
print(f"\n  Overall status: {'ALL FILES OK ✓' if all_ok else 'ERRORS DETECTED ✗'}")

# ── Index alignment spot-check ────────────────────────────────────────────────
print("\n[VERIFICATION] Spot-check: first label of each train split must match.")
lbl_raw     = pd.read_csv(raw_train_path,     encoding="utf-8-sig")["labels"].iloc[0]
lbl_clean   = pd.read_csv(clean_train_path,   encoding="utf-8-sig")["labels"].iloc[0]
lbl_stemmed = pd.read_csv(stemmed_train_path, encoding="utf-8-sig")["labels"].iloc[0]
match = lbl_raw == lbl_clean == lbl_stemmed
print(f"  train_raw[0].labels    = {lbl_raw}")
print(f"  train_clean[0].labels  = {lbl_clean}")
print(f"  train_stemmed[0].labels= {lbl_stemmed}")
print(f"  Labels match: {'YES ✓' if match else 'NO ✗ — check split logic'}")

print("\n[DONE] Preprocessing complete. Data is ready for modelling.")
print(f"       Output directory: {os.path.abspath(OUTPUT_DIR)}/")

  OUTPUT FILE SUMMARY
File                      Rows   Size (KB)  Status
------------------------------------------------------------
  train_raw               5065       738.7  ✓ OK
  val_raw                 1086       152.3  ✓ OK
  test_raw                1086       160.1  ✓ OK
  train_clean             5065       716.2  ✓ OK
  val_clean               1086       147.5  ✓ OK
  test_clean              1086       155.3  ✓ OK
  train_stemmed           5065       454.3  ✓ OK
  val_stemmed             1086        93.6  ✓ OK
  test_stemmed            1086        98.6  ✓ OK
------------------------------------------------------------

  Overall status: ALL FILES OK ✓

[VERIFICATION] Spot-check: first label of each train split must match.
  train_raw[0].labels    = 1
  train_clean[0].labels  = 1
  train_stemmed[0].labels= 1
  Labels match: YES ✓

[DONE] Preprocessing complete. Data is ready for modelling.
       Output directory: /content/data/
